In [1]:
# CSC202 AI — Sudoku Project Assignment
# Shared Data Models & Structures for Constraint Satisfaction Solvers

import copy

def calculate_cell_peers(row, col):
    # Aggregates unique coordinate keys within row, column, and local 3x3 block
    peer_set = set()
    
    # 1. Grab horizontal matches across row
    for target_c in range(9):
        if target_c != col:
            peer_set.add((row, target_c))
            
    # 2. Grab vertical matches down column
    for target_r in range(9):
        if target_r != row:
            peer_set.add((target_r, col))
            
    # 3. Grab block matches within local bounding box grid boundaries
    start_row_boundary = 3 * (row // 3)
    start_col_boundary = 3 * (col // 3)
    
    for r_offset in range(start_row_boundary, start_row_boundary + 3):
        for c_offset in range(start_col_boundary, start_col_boundary + 3):
            if (r_offset, c_offset) != (row, col):
                peer_set.add((r_offset, c_offset))
                
    return peer_set

# Initialize PEERS dictionary using standard nested iterations rather than comprehension shorthand
PEERS = {}
for r_idx in range(9):
    for c_idx in range(9):
        PEERS[(r_idx, c_idx)] = calculate_cell_peers(r_idx, c_idx)

# Initialize ARCS lookup tracking pairs manually
ARCS = []
for primary_cell in PEERS.keys():
    for neighbor_peer in PEERS[primary_cell]:
        ARCS.append((primary_cell, neighbor_peer))


class CSPState:
    """Tracks active value assignments and available solution domains globally across the grid."""
    
    def __init__(self, puzzle):
        self.assignment = {}
        self.domains = {}
        
        # Prepopulate all coordinate slots with full standard search pools
        for row in range(9):
            for col in range(9):
                default_pool = []
                for val in range(1, 10):
                    default_pool.append(val)
                self.domains[(row, col)] = set(default_pool)

        # Apply locked values dictated by the puzzle matrix setup layout
        for row in range(9):
            for col in range(9):
                if puzzle[row][col] != 0:
                    self.lock_initial_constraint((row, col), puzzle[row][col])

    def lock_initial_constraint(self, cell_coord, assigned_num):
        self.assignment[cell_coord] = assigned_num
        self.domains[cell_coord] = {assigned_num}

    def unassigned_vars(self):
        # Accumulates list of all tracking nodes that lack locked assignments
        open_slots = []
        for map_key in self.domains.keys():
            if map_key not in self.assignment:
                open_slots.append(map_key)
        return open_slots

    def is_complete(self):
        return len(self.assignment) == 81

    def is_consistent(self, cell_coord, prospective_val):
        # Verifies if selection conflicts with existing assignments in neighbors
        for peer_node in PEERS[cell_coord]:
            if peer_node in self.assignment:
                if self.assignment[peer_node] == prospective_val:
                    return False
        return True

    def to_grid(self):
        # Transforms map storage values back into structural nested output lists
        output_matrix = []
        for i in range(9):
            output_matrix.append([0] * 9)
            
        for (r, c), numerical_val in self.assignment.items():
            output_matrix[r][c] = numerical_val
        return output_matrix

    def copy(self):
        # High efficiency shallow object cloner structure with structural deep dictionary replication
        cloned_instance = object.__new__(CSPState)
        cloned_instance.assignment = dict(self.assignment)
        
        cloned_domains_map = {}
        for dictionary_key in self.domains.keys():
            cloned_domains_map[dictionary_key] = set(self.domains[dictionary_key])
            
        cloned_instance.domains = cloned_domains_map
        return cloned_instance


def check_solution_integrity(matrix_grid):
    # Validation engine verifying sudoku rules across rows, columns, and 3x3 zones
    expected_standard_set = set(range(1, 10))
    
    # 1. Evaluate Row Rows Integrity
    for r in range(9):
        if set(matrix_grid[r]) != expected_standard_set:
            return False
            
    # 2. Evaluate Column Columns Integrity
    for c in range(9):
        column_set = set()
        for r in range(9):
            column_set.add(matrix_grid[r][c])
        if column_set != expected_standard_set:
            return False
            
    # 3. Evaluate Sub-Box Areas Integrity
    for b_row in range(3):
        for b_col in range(3):
            subgrid_set = set()
            for row_offset in range(3):
                for col_offset in range(3):
                    val = matrix_grid[b_row * 3 + row_offset][b_col * 3 + col_offset]
                    subgrid_set.add(val)
            if subgrid_set != expected_standard_set:
                return False
                
    return True

In [2]:
import random

# Flat primitive array configs
CFG = {
    "Easy": 36,
    "Medium": 28,
    "Hard": 22,
    "Expert": 17
}

def check_placement_valid(g, r, c, val):
    # Quick linear validation scan across column entries
    if val in g[r]:
        return False
        
    for idx in range(9):
        if g[idx][c] == val:
            return False
            
    # Locate grid coordinates manually
    br = 3 * (r // 3)
    bc = 3 * (c // 3)
    
    for i in range(br, br + 3):
        for j in range(bc, bc + 3):
            if g[i][j] == val:
                return False
    return True

def populate_grid(g):
    # Standard linear search scanning to find unassigned coordinates
    for i in range(9):
        for j in range(9):
            if g[i][j] == 0:
                nums = [1, 2, 3, 4, 5, 6, 7, 8, 9]
                random.shuffle(nums)
                
                for candidate in nums:
                    if check_placement_valid(g, i, j, candidate):
                        g[i][j] = candidate
                        if populate_grid(g):
                            return True
                        g[i][j] = 0
                return False
    return True

def count_solutions(g, cap=2):
    for i in range(9):
        for j in range(9):
            if g[i][j] == 0:
                total = 0
                for d in range(1, 10):
                    if check_placement_valid(g, i, j, d):
                        g[i][j] = d
                        total += count_solutions(g, cap - total)
                        g[i][j] = 0
                        if total >= cap:
                            return total
                return total
    return 1

def generate_puzzle(difficulty="Medium", seed=None):
    if seed is not None:
        random.seed(seed)

    # Manual iterative zero allocation for matrix setups
    sol = []
    for _ in range(9):
        sol.append([0, 0, 0, 0, 0, 0, 0, 0, 0])
        
    populate_grid(sol)

    hints = CFG.get(difficulty, 28)
    
    # Flat linear tracking array loop for all index pairings
    cells = []
    for r in range(9):
        for c in range(9):
            cells.append((r, c))
            
    random.shuffle(cells)

    # Row slicing comprehensions to substitute deepcopy references
    puz = [row[:] for row in sol]
    rem = 0
    limit = 81 - hints

    for r, c in cells:
        if rem >= limit:
            break
            
        old = puz[r][c]
        puz[r][c] = 0
        
        # Test copy allocation matrix build manually
        t_copy = [row[:] for row in puz]
        if count_solutions(t_copy) == 1:
            rem += 1
        else:
            puz[r][c] = old

    return puz, sol

def display_grid(g, title_header=""):
    if title_header:
        print("\n--- " + str(title_header) + " ---")
        
    print("+-------+-------+-------+")
    for i in range(9):
        row_out = "| "
        for j in range(9):
            val = g[i][j]
            char = "." if val == 0 else str(val)
            row_out += char + " "
            
            if j == 2 or j == 5:
                row_out += "| "
                
        row_out += "|"
        print(row_out)
        
        if i == 2 or i == 5:
            print("+-------+-------+-------+")
            
    print("+-------+-------+-------+")

In [3]:
# CSC202 AI — Sudoku Project Assignment
# Arc Consistency (AC-3) Solver Module with MRV and Degree Heuristics

import copy
import time
from collections import deque

def run_ac3_algorithm(cell_domains):
    # Enforces arc consistency constraints using a work-list queue
    work_queue = deque(ARCS)
    
    while len(work_queue) > 0:
        target_cell, neighbor_cell = work_queue.popleft()
        
        if prune_unsupported_values(cell_domains, target_cell, neighbor_cell):
            # If a cell domain is completely empty, the path is invalid
            if len(cell_domains[target_cell]) == 0:
                return False  
                
            # Re-evaluate all other connected arcs affected by the domain shrink
            for peer in PEERS[target_cell]:
                if peer != neighbor_cell:
                    work_queue.append((peer, target_cell))
                    
    return True

def prune_unsupported_values(cell_domains, cell_x, cell_y):
    # Removes values from domain of x if they have no valid match left in domain of y
    is_revised = False
    current_values = list(cell_domains[cell_x])
    
    for x_val in current_values:
        # Check if there is any option left in cell y that isn't equal to x_val
        has_matching_option = False
        for y_val in cell_domains[cell_y]:
            if y_val != x_val:
                has_matching_option = True
                break
                
        if not has_matching_option:
            cell_domains[cell_x].discard(x_val)
            is_revised = True
            
    return is_revised

def choose_variable_mrv_degree(state):
    # Variable selection choosing Lowest Domain Size (MRV) breaking ties with Maximum Unassigned Degree
    unfilled_vars = state.unassigned_vars()
    
    best_variable = None
    smallest_domain_size = 999999
    highest_degree_count = -1
    
    for current_var in unfilled_vars:
        domain_len = len(state.domains[current_var])
        
        # Calculate active constraint degree explicitly
        degree_len = 0
        for peer in PEERS[current_var]:
            if peer not in state.assignment:
                degree_len += 1
        
        # Apply MRV comparison checks with explicit Degree tie-breaking logic
        if domain_len < smallest_domain_size:
            smallest_domain_size = domain_len
            highest_degree_count = degree_len
            best_variable = current_var
        elif domain_len == smallest_domain_size:
            if degree_len > highest_degree_count:
                highest_degree_count = degree_len
                best_variable = current_var
                
    return best_variable

def solve_with_ac3(puzzle):
    current_state = CSPState(puzzle)
    tracking_stats = {"explored": 0, "reverted": 0}

    # Initial preprocessing arc check pass
    if not run_ac3_algorithm(current_state.domains):
        return {
            "algorithm": "AC-3 + MRV + Degree",
            "solved": False, 
            "solution": None,
            "time_taken": 0.0, 
            "states": 0, 
            "backtracks": 0
        }

    fill_obvious_singles(current_state)

    start_timer = time.perf_counter()
    was_successful = backtrack_ac3_driver(current_state, tracking_stats)
    elapsed_time = time.perf_counter() - start_timer

    final_solution_grid = None
    if was_successful:
        final_solution_grid = current_state.to_grid()

    return {
        "algorithm": "AC-3 + MRV + Degree",
        "solved": was_successful,
        "solution": final_solution_grid,
        "time_taken": elapsed_time,
        "states": tracking_stats["explored"],
        "backtracks": tracking_stats["reverted"]
    }

def fill_obvious_singles(state):
    # Automatically assigns values to coordinates containing exactly one possibility left
    for current_key in state.domains.keys():
        domain_pool = state.domains[current_key]
        if len(domain_pool) == 1:
            if current_key not in state.assignment:
                for element in domain_pool:
                    state.assignment[current_key] = element

def backtrack_ac3_driver(state, tracking_stats):
    if state.is_complete():
        return True

    active_var = choose_variable_mrv_degree(state)
    sorted_values = sorted(list(state.domains[active_var]))

    for active_val in sorted_values:
        tracking_stats["explored"] += 1
        
        if not state.is_consistent(active_var, active_val):
            continue

        # Create localized deep copies of configurations to allow complete fallbacks
        saved_domains_map = {}
        for key in state.domains.keys():
            saved_domains_map[key] = set(state.domains[key])
            
        saved_assignments_map = dict(state.assignment)

        # Apply choice paths
        state.assignment[active_var] = active_val
        state.domains[active_var] = {active_val}

        # Enforce inference checks downstream
        if run_ac3_algorithm(state.domains):
            fill_obvious_singles(state)
            if backtrack_ac3_driver(state, tracking_stats):
                return True

        # Revert changes entirely if sub-tree triggers branch failures
        tracking_stats["reverted"] += 1
        state.domains = saved_domains_map
        state.assignment = saved_assignments_map

    return False


In [4]:

# CSC202 AI — Sudoku Project Assignment
# Standard Baseline Depth-First Search (DFS) Backtracking Solver

import copy
import time

def solve_with_pure_backtracking(puzzle):
    # Initializes a fresh CSP instance and runs baseline DFS solver logic
    current_state = CSPState(puzzle)
    tracking_metrics = {"explored": 0, "reverted": 0}

    start_timer = time.perf_counter()
    was_successful = execute_dfs_backtrack(current_state, tracking_metrics)
    elapsed_time = time.perf_counter() - start_timer

    final_solution_grid = None
    if was_successful:
        final_solution_grid = current_state.to_grid()

    return {
        "algorithm": "Backtracking (DFS)",
        "solved": was_successful,
        "solution": final_solution_grid,
        "time_taken": elapsed_time,
        "states": tracking_metrics["explored"],
        "backtracks": tracking_metrics["reverted"]
    }

def execute_dfs_backtrack(state, metrics):
    # Standard exhaustive backtracking without advanced filtering or choice sorting
    if state.is_complete():
        return True

    # Pull the first open coordinate from our list array
    all_open_variables = state.unassigned_vars()
    target_var = all_open_variables[0]

    # Iteratively evaluate numbers from 1 to 9
    for candidate_value in range(1, 10):
        metrics["explored"] += 1
        
        if state.is_consistent(target_var, candidate_value):
            # Assign tentative option value labels
            state.assignment[target_var] = candidate_value
            state.domains[target_var] = {candidate_value}

            # Dive deeper recursively along the current choice path
            if execute_dfs_backtrack(state, metrics):
                return True

            # If the path hits a dead-end, step backward and clear state flags
            metrics["reverted"] += 1
            if target_var in state.assignment:
                del state.assignment[target_var]
                
            # Re-initialize the domain pool back to default full state range options
            default_pool = []
            for num in range(1, 10):
                default_pool.append(num)
            state.domains[target_var] = set(default_pool)

    return False


In [5]:
# CSC202 AI — Sudoku Project Assignment
# Forward Checking (FC) Engine with MRV and LCV implementation rules

import time

# ------------------------------------------------------------------------------
# 1. Look-Ahead Filtering Rules
# ------------------------------------------------------------------------------

def apply_forward_checking(grid_domains, chosen_cell, chosen_val, current_board):
    # Dynamically drops current value option from all neighbor cell pools
    history_of_drops = {}
    
    for relative_neighbor in PEERS[chosen_cell]:
        if relative_neighbor in current_board:
            continue
            
        if chosen_val in grid_domains[relative_neighbor]:
            grid_domains[relative_neighbor].discard(chosen_val)
            history_of_drops[relative_neighbor] = chosen_val
            
            # Catch immediate domain washouts early
            if len(grid_domains[relative_neighbor]) == 0:
                # Rollback current iteration removals prior to exit sequence
                for modified_node in history_of_drops.keys():
                    original_digit = history_of_drops[modified_node]
                    grid_domains[modified_node].add(original_digit)
                return None  
                
    return history_of_drops

def reverse_forward_checks(grid_domains, history_of_drops):
    # Restores state domain settings using historic tracking dictionary maps
    for modified_node in history_of_drops.keys():
        original_digit = history_of_drops[modified_node]
        grid_domains[modified_node].add(original_digit)

# ------------------------------------------------------------------------------
# 2. Variable and Value Ordering Lookups
# ------------------------------------------------------------------------------

def get_variable_via_mrv(state):
    # Identifies active coordinates displaying the lowest search choice options
    unfilled_vars = state.unassigned_vars()
    selected_variable = None
    minimum_choices = 999999
    
    for node in unfilled_vars:
        current_choices_len = len(state.domains[node])
        if current_choices_len < minimum_choices:
            minimum_choices = current_choices_len
            selected_variable = node
            
    return selected_variable

def get_lcv_sorted_values(state, cell_node):
    # Prioritizes options that maintain maximal flexibility for adjacent nodes
    candidate_values = list(state.domains[cell_node])
    impact_metrics_list = []
    
    for candidate in candidate_values:
        constraint_impact_score = 0
        for target_peer in PEERS[cell_node]:
            if target_peer not in state.assignment:
                if candidate in state.domains[target_peer]:
                    constraint_impact_score += 1
        impact_metrics_list.append((constraint_impact_score, candidate))
        
    # Order ascendingly (lowest impact values get tested first)
    impact_metrics_list.sort()
    
    final_ordered_sequence = []
    for scoring_record in impact_metrics_list:
        final_ordered_sequence.append(scoring_record[1])
        
    return final_ordered_sequence

# ------------------------------------------------------------------------------
# 3. Main Operational Solver Routines
# ------------------------------------------------------------------------------

def solve_with_forward_checking(puzzle):
    runtime_state = CSPState(puzzle)
    run_metrics = {"explored": 0, "reverted": 0}

    timer_mark = time.perf_counter()
    is_successful = run_fc_backtrack_loop(runtime_state, run_metrics)
    total_execution_time = time.perf_counter() - timer_mark

    solved_matrix_output = None
    if is_successful:
        solved_matrix_output = runtime_state.to_grid()

    return {
        "algorithm": "Forward Checking (FC)",
        "solved": is_successful,
        "solution": solved_matrix_output,
        "time_taken": total_execution_time,
        "states": run_metrics["explored"],
        "backtracks": run_metrics["reverted"]
    }

def run_fc_backtrack_loop(state, counter_tracker):
    # Main recursive driver routine enforcing local constraint optimizations
    if state.is_complete():
        return True

    target_var = get_variable_via_mrv(state)
    ordered_values = get_lcv_sorted_values(state, target_var)

    for test_value in ordered_values:
        counter_tracker["explored"] += 1

        if not state.is_consistent(target_var, test_value):
            continue

        # Cache point adjustments before locking local variables down
        state.assignment[target_var] = test_value
        saved_domain_backup = set(state.domains[target_var])
        state.domains[target_var] = {test_value}

        # Apply constraint screening lookaheads across adjacent open peers
        filtering_records = apply_forward_checking(state.domains, target_var, test_value, state.assignment)

        if filtering_records != None:
            if run_fc_backtrack_loop(state, counter_tracker):
                return True
            # Revert peer adjustments if downstream tree returns failures
            reverse_forward_checks(state.domains, filtering_records)

        # Clear active flags and shift state configurations backward on dead-ends
        counter_tracker["reverted"] += 1
        if target_var in state.assignment:
            del state.assignment[target_var]
        state.domains[target_var] = saved_domain_backup

    return False

In [6]:
# CSC202 AI — Sudoku Project Assignment
# Simulated Annealing (SA) Local Search Engine Optimization

import copy
import math
import random
import time

def evaluate_grid_conflicts(matrix):
    # Calculates row and column constraint duplication penalties
    clash_score = 0
    
    for row_idx in range(9):
        row_frequencies = [0] * 10
        col_frequencies = [0] * 10
        
        for col_idx in range(9):
            row_digit = matrix[row_idx][col_idx]
            col_digit = matrix[col_idx][row_idx]
            
            if row_digit != 0:
                row_frequencies[row_digit] += 1
            if col_digit != 0:
                col_frequencies[col_digit] += 1
                
        # Aggregate overlapping instance violations
        for value in range(1, 10):
            if row_frequencies[value] > 1:
                clash_score += (row_frequencies[value] - 1)
            if col_frequencies[value] > 1:
                clash_score += (col_frequencies[value] - 1)
                
    return clash_score

def create_randomized_boxes(puzzle):
    # Fills out missing digits block-by-block to guarantee subgrid validity upfront
    working_grid = copy.deepcopy(puzzle)
    
    for block_r in range(3):
        for block_c in range(3):
            values_in_block = set()
            unfilled_coordinates = []
            
            for offset_r in range(3):
                for offset_c in range(3):
                    actual_r = block_r * 3 + offset_r
                    actual_c = block_c * 3 + offset_c
                    cell_val = working_grid[actual_r][actual_c]
                    
                    if cell_val != 0:
                        values_in_block.add(cell_val)
                    else:
                        unfilled_coordinates.append((actual_r, actual_c))
            
            # Find missing items for this 3x3 block manually
            missing_digits = []
            for candidate in range(1, 10):
                if candidate not in values_in_block:
                    missing_digits.append(candidate)
                    
            random.shuffle(missing_digits)
            
            # Populate empty slots sequentially
            for marker in range(len(unfilled_coordinates)):
                target_r, target_c = unfilled_coordinates[marker]
                working_grid[target_r][target_c] = missing_digits[marker]
                
    return working_grid

def solve_with_simulated_annealing(puzzle, total_restarts=5, steps_per_run=200000, init_temp=2.0, floor_temp=0.01, cool_rate=0.9999):
    # Locate frozen default cells given by the puzzle layout
    immutable_nodes = set()
    for r in range(9):
        for c in range(9):
            if puzzle[r][c] != 0:
                immutable_nodes.add((r, c))
    
    # Map out non-fixed cells available for intra-block shuffling
    shufflable_blocks = {}
    for block_r in range(3):
        for block_c in range(3):
            valid_targets = []
            for offset_r in range(3):
                for offset_c in range(3):
                    cell_coords = (block_r * 3 + offset_r, block_c * 3 + offset_c)
                    if cell_coords not in immutable_nodes:
                        valid_targets.append(cell_coords)
                        
            if len(valid_targets) >= 2:
                shufflable_blocks[(block_r, block_c)] = valid_targets

    total_evaluations = 0
    uphill_moves_counter = 0
    start_time_marker = time.perf_counter()
    available_block_keys = list(shufflable_blocks.keys())

    for active_restart in range(total_restarts):
        current_board = create_randomized_boxes(puzzle)
        current_cost = evaluate_grid_conflicts(current_board)
        system_temp = init_temp
        
        for dynamic_step in range(steps_per_run):
            if current_cost == 0:
                elapsed_run_time = time.perf_counter() - start_time_marker
                return {
                    "algorithm": "Simulated Annealing",
                    "solved": True,
                    "solution": current_board,
                    "time_taken": elapsed_run_time,
                    "states": total_evaluations + dynamic_step,
                    "backtracks": uphill_moves_counter
                }

            # Select a random subgrid block and isolate two mutable elements
            chosen_block = random.choice(available_block_keys)
            target_pool = shufflable_blocks[chosen_block]
            node_a, node_b = random.sample(target_pool, 2)

            # Perform values swap transformation manually
            temp_swap = current_board[node_a[0]][node_a[1]]
            current_board[node_a[0]][node_a[1]] = current_board[node_b[0]][node_b[1]]
            current_board[node_b[0]][node_b[1]] = temp_swap

            updated_cost = evaluate_grid_conflicts(current_board)
            cost_difference = updated_cost - current_cost
            total_evaluations += 1

            approve_mutation = False
            if cost_difference < 0:
                approve_mutation = True
            else:
                adjusted_temp = system_temp
                if adjusted_temp < 1e-10:
                    adjusted_temp = 1e-10
                
                # Probability distribution limit boundary check
                prob_threshold = math.exp(-cost_difference / adjusted_temp)
                if random.random() < prob_threshold:
                    approve_mutation = True

            if approve_mutation:
                current_cost = updated_cost
                if cost_difference > 0:
                    uphill_moves_counter += 1
            else:
                # Cancel operation and revert board alignment values manually
                temp_swap = current_board[node_a[0]][node_a[1]]
                current_board[node_a[0]][node_a[1]] = current_board[node_b[0]][node_b[1]]
                current_board[node_b[0]][node_b[1]] = temp_swap

            # Apply geometric scheduling cooling step adjustments
            system_temp = system_temp * cool_rate
            if system_temp < floor_temp:
                system_temp = floor_temp

        total_evaluations += steps_per_run

    time_elapsed = time.perf_counter() - start_time_marker
    return {
        "algorithm": "Simulated Annealing",
        "solved": False,
        "solution": None,
        "time_taken": time_elapsed,
        "states": total_evaluations,
        "backtracks": uphill_moves_counter
    }

In [7]:


import threading
import time


class SolverThread(threading.Thread):
    def __init__(self, name, solver, board):
        super().__init__()
        self.name = name
        self.solver = solver
        self.board = board

        self.result = None
        self.error = None

    def run(self):
        try:
            self.result = self.solver(self.board)
        except Exception as e:
            self.error = e


def execute_race_mode(
    puzzle,
    algo_a,
    algo_b,
    name_a="Agent A",
    name_b="Agent B",
    max_seconds=120,
):

    # Give each solver its own copy of the board
    board_a = [row[:] for row in puzzle]
    board_b = [row[:] for row in puzzle]

    t1 = SolverThread(name_a, algo_a, board_a)
    t2 = SolverThread(name_b, algo_b, board_b)

    t1.start()
    t2.start()

    t1.join(max_seconds)
    t2.join(max_seconds)

    result_a = t1.result
    result_b = t2.result

    winner = "Neither (Timeout/Failed)"
    delta = None

    solved_a = result_a and result_a.get("solved")
    solved_b = result_b and result_b.get("solved")

    if solved_a and solved_b:
        if result_a["time_taken"] < result_b["time_taken"]:
            winner = name_a
            delta = result_b["time_taken"] - result_a["time_taken"]
        else:
            winner = name_b
            delta = result_a["time_taken"] - result_b["time_taken"]

    elif solved_a:
        winner = name_a

    elif solved_b:
        winner = name_b

    print()
    print("=" * 66)
    print("                ADVERSARIAL COMPETE MODE")
    print("=" * 66)
    print("Agent                    Time       States   Backtracks")
    print("-" * 66)

    if result_a:
        print(
            f"{name_a:<24}"
            f"{result_a['time_taken']:>8.3f}s"
            f"{result_a.get('states', 0):>12,}"
            f"{result_a.get('backtracks', 0):>13,}"
        )
    else:
        print(f"{name_a:<24}{'TIMEOUT':>8}")

    if result_b:
        print(
            f"{name_b:<24}"
            f"{result_b['time_taken']:>8.3f}s"
            f"{result_b.get('states', 0):>12,}"
            f"{result_b.get('backtracks', 0):>13,}"
        )
    else:
        print(f"{name_b:<24}{'TIMEOUT':>8}")

    print("=" * 66)

    if delta is not None:
        print(f"Winner: {winner} (+{delta:.4f}s)")
    else:
        print(f"Winner: {winner}")

    return {
        "winner": winner,
        "delta": delta,
        "agents": [
            {"name": name_a, "result": result_a},
            {"name": name_b, "result": result_b},
        ],
    }

import csv

def check_solution_integrity(grid):
    required = set(range(1, 10))
    for i in range(9):
        if set(grid[i]) != required: return False
        if {grid[r][i] for r in range(9)} != required: return False
    for br in range(3):
        for bc in range(3):
            box = {grid[br*3+dr][bc*3+dc] for dr in range(3) for dc in range(3)}
            if box != required: return False
    return True

def display_single_level_table(results, level): pass
def display_global_comparison_report(results): pass
def export_results_to_csv_file(results, path): pass

ALGORITHMS_MAP = {
    "Backtracking DFS":  solve_with_pure_backtracking,
    "AC-3 + MRV":        solve_with_ac3,
    "Simulated Annealing": solve_with_simulated_annealing,
    "Forward Checking":  solve_with_forward_checking,
}
ALGO_LIST = list(ALGORITHMS_MAP.keys())
DIFFICULTIES = ["Easy", "Medium", "Hard", "Expert"]


In [10]:
import copy
import ipywidgets as w
from IPython.display import display, HTML, clear_output
from html import escape

# ── state ──────────────────────────────────────────────────────────────────
_state = {"puzzle": None}

ALGO_MAP = {
    "Backtracking DFS":    solve_with_pure_backtracking,
    "AC-3 + MRV":          solve_with_ac3,
    "Forward Checking":    solve_with_forward_checking,
    "Simulated Annealing": solve_with_simulated_annealing,
}
ALGO_NAMES  = list(ALGO_MAP.keys())
DIFF_LEVELS = ["Easy", "Medium", "Hard", "Expert"]

# ── board renderer ─────────────────────────────────────────────────────────
def _board_html(puzzle, solution=None, title=""):
    given = {(r,c) for r in range(9) for c in range(9) if puzzle[r][c]}
    grid  = solution if solution else puzzle
    h = '<div style="text-align:center;font-family:monospace;">'
    if title:
        h += f'<div style="color:#7c6af7;font-size:15px;font-weight:700;margin-bottom:10px;">{escape(title)}</div>'
    h += '<table style="border-collapse:collapse;margin:0 auto;">'
    for r in range(9):
        h += "<tr>"
        for c in range(9):
            # cell type
            if   (r,c) in given: bg,fg = "#191730","#b8b0ff"
            elif grid[r][c]:     bg,fg = "#0d2419","#6af7c4"
            else:                bg,fg = "#13131e","#2a2a42"
            val = str(grid[r][c]) if grid[r][c] else "·"
            # borders
            bt = "3px solid #7c6af7" if r==0 else ("3px solid #7c6af7" if r in(3,6) else "1px solid #2a2a42")
            bb = "3px solid #7c6af7" if r==8 else ("3px solid #7c6af7" if r in(2,5) else "1px solid #2a2a42")
            bl = "3px solid #7c6af7" if c==0 else ("3px solid #7c6af7" if c in(3,6) else "1px solid #2a2a42")
            br_ = "3px solid #7c6af7" if c==8 else ("3px solid #7c6af7" if c in(2,5) else "1px solid #2a2a42")
            h += (f'<td style="width:44px;height:44px;text-align:center;font-size:18px;font-weight:700;' 
                  f'background:{bg};color:{fg};border-top:{bt};border-bottom:{bb};' 
                  f'border-left:{bl};border-right:{br_};">{val}</td>')
        h += "</tr>"
    h += "</table></div>"
    return h

def _metrics_html(r):
    ms   = r.get("time_taken", 0) * 1000
    t    = f"{ms:.3f} ms" if ms < 1 else (f"{ms:.1f} ms" if ms < 1000 else f"{ms/1000:.2f} s")
    ok   = r.get("solved", False)
    pill = ('<span style="background:#0d2419;color:#6af7c4;border:1px solid #6af7c4;'
            'padding:2px 10px;border-radius:20px;font-size:12px;font-weight:700;">SOLVED</span>')
    if not ok:
        pill = ('<span style="background:#2a0a14;color:#f76a8f;border:1px solid #f76a8f;'
                'padding:2px 10px;border-radius:20px;font-size:12px;font-weight:700;">FAILED</span>')
    return f"""
    <div style="display:flex;gap:10px;margin-top:14px;flex-wrap:wrap;font-family:monospace;">
      <div style="flex:1;min-width:100px;background:#13131e;border:1px solid #2a2a42;border-radius:10px;padding:11px 14px;">
        <div style="font-size:10px;color:#6b6b8a;letter-spacing:2px;">TIME</div>
        <div style="font-size:20px;font-weight:700;color:#7c6af7;">{t}</div>
      </div>
      <div style="flex:1;min-width:100px;background:#13131e;border:1px solid #2a2a42;border-radius:10px;padding:11px 14px;">
        <div style="font-size:10px;color:#6b6b8a;letter-spacing:2px;">STATES</div>
        <div style="font-size:20px;font-weight:700;color:#f7c46a;">{r.get("states",0):,}</div>
      </div>
      <div style="flex:1;min-width:100px;background:#13131e;border:1px solid #2a2a42;border-radius:10px;padding:11px 14px;">
        <div style="font-size:10px;color:#6b6b8a;letter-spacing:2px;">BACKTRACKS</div>
        <div style="font-size:20px;font-weight:700;color:#f76a8f;">{r.get("backtracks",0):,}</div>
      </div>
      <div style="flex:1;min-width:100px;background:#13131e;border:1px solid #2a2a42;border-radius:10px;padding:11px 14px;">
        <div style="font-size:10px;color:#6b6b8a;letter-spacing:2px;">RESULT</div>
        <div style="margin-top:6px;">{pill}</div>
      </div>
    </div>"""

def _compare_html(results, title=""):
    solved  = [r for r in results if r.get("solved")]
    min_t   = min((r["time_taken"] for r in solved), default=0)
    max_t   = max((r["time_taken"] for r in results), default=1) or 1
    max_s   = max((r.get("states",0) for r in results), default=1) or 1
    max_b   = max((r.get("backtracks",0) for r in results), default=1) or 1
    clrs    = ["#7c6af7","#6af7c4","#f76a8f","#f7c46a"]
    ms = lambda s: (f"{s*1000:.3f} ms" if s*1000<1 else (f"{s*1000:.1f} ms" if s*1000<1000 else f"{s:.2f} s"))
    h  = f'<div style="font-family:monospace;font-size:14px;color:#f7c46a;margin-bottom:10px;">{escape(title)}</div>'
    h += '<table style="border-collapse:collapse;width:100%;">'
    h += '<tr style="border-bottom:2px solid #2a2a42;">'
    for col in ["Algorithm","Time","States","Backtracks","Solved"]:
        h += f'<th style="background:#13131e;color:#6b6b8a;padding:9px 12px;text-align:left;font-size:10px;letter-spacing:2px;">{col.upper()}</th>'
    h += "</tr>"
    for i,r in enumerate(results):
        is_w  = r.get("solved") and r["time_taken"]==min_t
        clr   = "#6af7c4" if is_w else "#e8e8f2"
        tp    = r["time_taken"]/max_t*100
        sp    = r.get("states",0)/max_s*100
        bp    = r.get("backtracks",0)/max_b*100
        sym   = '<span style="color:#6af7c4;font-weight:700;">✓ Yes</span>' if r.get("solved") else '<span style="color:#f76a8f;font-weight:700;">✗ No</span>'
        trp   = "🏆 " if is_w else ""
        bar   = lambda pct,c: f'<span style="background:#1e1e2e;border-radius:3px;height:11px;width:120px;display:inline-block;vertical-align:middle;"><span style="height:100%;border-radius:3px;display:inline-block;background:{c};width:{pct:.0f}%;"></span></span> '
        h += f'<tr style="border-bottom:1px solid #1e1e2e;color:{clr};">'
        h += f'<td style="padding:10px 12px;font-weight:{700 if is_w else 400};">{trp}{escape(r.get("algorithm",""))}</td>'
        h += f'<td style="padding:10px 12px;">{bar(tp,clrs[i])}{ms(r["time_taken"])}</td>'
        h += f'<td style="padding:10px 12px;">{bar(sp,clrs[i])}{r.get("states",0):,}</td>'
        h += f'<td style="padding:10px 12px;">{bar(bp,clrs[i])}{r.get("backtracks",0):,}</td>'
        h += f'<td style="padding:10px 12px;">{sym}</td></tr>'
    h += "</table>"
    return h

def _race_html(ra, rb, na, nb_):
    sa = ra.get("solved") if ra else False
    sb = rb.get("solved") if rb else False
    ms = lambda s: (f"{s*1000:.3f} ms" if s*1000<1 else (f"{s*1000:.1f} ms" if s*1000<1000 else f"{s:.2f} s"))
    if sa and sb:
        if ra["time_taken"] <= rb["time_taken"]: winner,margin = na, rb["time_taken"]-ra["time_taken"]
        else:                                    winner,margin = nb_,ra["time_taken"]-rb["time_taken"]
        verdict = f"🏆 Winner: {escape(winner)}  (faster by {ms(margin)})"
    elif sa: verdict = f"🏆 Winner: {escape(na)}"
    elif sb: verdict = f"🏆 Winner: {escape(nb_)}"
    else:    verdict = "Neither agent solved the puzzle."
    def agent(name,r,clr):
        if not r: return f'<div style="flex:1;background:#0e0e1a;border-radius:8px;padding:12px;border:1px solid #2a2a42;font-family:monospace;"><b style="color:{clr};">{escape(name)}</b><br>No result</div>'
        ok   = r.get("solved")
        pill = (f'<span style="background:#0d2419;color:#6af7c4;border:1px solid #6af7c4;padding:2px 8px;border-radius:20px;font-size:11px;">SOLVED</span>' if ok
                else '<span style="background:#2a0a14;color:#f76a8f;border:1px solid #f76a8f;padding:2px 8px;border-radius:20px;font-size:11px;">FAILED</span>')
        return (f'<div style="flex:1;background:#0e0e1a;border-radius:8px;padding:12px;border:1px solid #2a2a42;font-family:monospace;font-size:13px;">' 
                f'<div style="color:{clr};font-weight:700;font-size:14px;margin-bottom:8px;">{escape(name)}</div>' 
                f'Time: <b>{ms(r["time_taken"])}</b><br>States: <b>{r.get("states",0):,}</b><br>Backtracks: <b>{r.get("backtracks",0):,}</b><br><div style="margin-top:6px;">{pill}</div></div>')
    return (f'<div style="background:#13131e;border:1px solid #2a2a42;border-radius:12px;padding:16px;margin-top:10px;">' 
            f'<div style="color:#f76a8f;font-size:14px;font-weight:700;margin-bottom:10px;font-family:monospace;">⚔  ADVERSARIAL RACE MODE</div>' 
            f'<div style="display:flex;gap:12px;">{agent(na,ra,"#7c6af7")}{agent(nb_,rb,"#6af7c4")}</div>' 
            f'<div style="margin-top:12px;padding:10px 14px;background:#0d2419;border-radius:8px;color:#6af7c4;font-weight:700;font-family:monospace;">{verdict}</div></div>')

# ── widgets ─────────────────────────────────────────────────────────────────
S   = {"description_width":"90px"}
BT  = w.Layout(width="152px",height="36px")
BT2 = w.Layout(width="172px",height="36px")

diff_dd  = w.Dropdown(options=DIFF_LEVELS, value="Medium",      description="Difficulty:", style=S, layout=w.Layout(width="205px"))
algo_dd  = w.Dropdown(options=ALGO_NAMES,  value="AC-3 + MRV",  description="Algorithm:",  style=S, layout=w.Layout(width="250px"))
agent_a  = w.Dropdown(options=ALGO_NAMES,  value=ALGO_NAMES[0], description="Agent A:",    style=S, layout=w.Layout(width="250px"))
agent_b  = w.Dropdown(options=ALGO_NAMES,  value=ALGO_NAMES[1], description="Agent B:",    style=S, layout=w.Layout(width="250px"))

btn_gen    = w.Button(description="⚡ Generate",    button_style="primary", layout=BT)
btn_solve  = w.Button(description="▶  Solve",       button_style="success", layout=BT)
btn_bench  = w.Button(description="📊 Compare All", button_style="warning", layout=BT2)
btn_levels = w.Button(description="🌐 All Levels",  button_style="info",    layout=BT2)
btn_race   = w.Button(description="⚔  Race!",       button_style="danger",  layout=BT)

out_board   = w.Output()
out_metrics = w.Output()
out_compare = w.Output()
out_race    = w.Output()

tab = w.Tab(children=[
    w.VBox([out_board, out_metrics]),
    w.VBox([out_compare]),
    w.VBox([
        w.HBox([agent_a, agent_b, btn_race],
               layout=w.Layout(align_items="center",gap="8px",margin="8px 0")),
        out_race
    ])
])
tab.set_title(0, "🎯 Solve")
tab.set_title(1, "📊 Compare")
tab.set_title(2, "⚔ Race")

# ── callbacks ───────────────────────────────────────────────────────────────
def on_generate(_):
    with out_board:
        clear_output(wait=True)
        display(HTML('<p style="text-align:center;color:#6b6b8a;font-family:monospace;">Generating...</p>'))
    with out_metrics: clear_output()
    try:
        _state["puzzle"], _ = generate_puzzle(diff_dd.value)
        n = sum(v for row in _state["puzzle"] for v in row if v)
        with out_board:
            clear_output(wait=True)
            display(HTML(_board_html(_state["puzzle"], title=f"{diff_dd.value} Puzzle  ·  {n} givens")))
    except Exception as e:
        with out_board:
            clear_output(wait=True)
            display(HTML(f'<p style="color:#f76a8f;font-family:monospace;">Error: {e}</p>'))

def on_solve(_):
    if not _state["puzzle"]:
        with out_metrics:
            clear_output()
            display(HTML('<p style="color:#f76a8f;font-family:monospace;">Generate a puzzle first.</p>'))
        return
    with out_board:
        clear_output(wait=True)
        display(HTML('<p style="text-align:center;color:#6b6b8a;font-family:monospace;">Solving...</p>'))
    try:
        result = ALGO_MAP[algo_dd.value](copy.deepcopy(_state["puzzle"]))
        label  = "SOLVED" if result["solved"] else "FAILED"
        with out_board:
            clear_output(wait=True)
            display(HTML(_board_html(_state["puzzle"], result.get("solution"), title=f"{algo_dd.value}  ·  {label}")))
        with out_metrics:
            clear_output(wait=True)
            display(HTML(_metrics_html(result)))
        tab.selected_index = 0
    except Exception as e:
        with out_board:
            clear_output(wait=True)
            display(HTML(f'<p style="color:#f76a8f;font-family:monospace;">Error: {e}</p>'))

def on_bench(_):
    if not _state["puzzle"]:
        with out_compare:
            clear_output()
            display(HTML('<p style="color:#f76a8f;font-family:monospace;">Generate a puzzle first.</p>'))
        return
    with out_compare:
        clear_output(wait=True)
        display(HTML('<p style="color:#6b6b8a;font-family:monospace;">Running all 4 algorithms...</p>'))
    results = [ALGO_MAP[a](copy.deepcopy(_state["puzzle"])) for a in ALGO_NAMES]
    with out_compare:
        clear_output(wait=True)
        display(HTML(_compare_html(results, f"Algorithm Comparison  ·  {diff_dd.value}")))
    tab.selected_index = 1

def on_all_levels(_):
    with out_compare:
        clear_output(wait=True)
        display(HTML('<p style="color:#6b6b8a;font-family:monospace;">Running all levels — please wait (1–2 min)...</p>'))
    html = ""
    for diff in DIFF_LEVELS:
        puz, _ = generate_puzzle(diff)
        res    = [ALGO_MAP[a](copy.deepcopy(puz)) for a in ALGO_NAMES]
        html  += _compare_html(res, f"Difficulty: {diff}")
        html  += "<hr style='border:none;border-top:1px solid #2a2a42;margin:16px 0;'>"
    with out_compare:
        clear_output(wait=True)
        display(HTML(html))
    tab.selected_index = 1

def on_race(_):
    if not _state["puzzle"]:
        with out_race:
            clear_output()
            display(HTML('<p style="color:#f76a8f;font-family:monospace;">Generate a puzzle first.</p>'))
        return
    na, nb_ = agent_a.value, agent_b.value
    if na == nb_:
        with out_race:
            clear_output()
            display(HTML('<p style="color:#f76a8f;font-family:monospace;">Pick two different algorithms.</p>'))
        return
    with out_race:
        clear_output(wait=True)
        display(HTML('<p style="color:#6b6b8a;font-family:monospace;">Racing...</p>'))
    ra = ALGO_MAP[na](copy.deepcopy(_state["puzzle"]))
    rb = ALGO_MAP[nb_](copy.deepcopy(_state["puzzle"]))
    with out_race:
        clear_output(wait=True)
        display(HTML(_race_html(ra, rb, na, nb_)))
    tab.selected_index = 2

btn_gen.on_click(on_generate)
btn_solve.on_click(on_solve)
btn_bench.on_click(on_bench)
btn_levels.on_click(on_all_levels)
btn_race.on_click(on_race)

# ── display ──────────────────────────────────────────────────────────────────
display(HTML("""
<div style="text-align:center; padding:10px 0 16px 0;">
  <span style="font-family:monospace; font-size:26px; font-weight:700; color:#e8e8f2;">
     <span style="color:#7c6af7;">Sudoku Solver</span> 
  </span>
</div>
"""))

display(w.VBox([
    w.HBox([diff_dd, algo_dd, btn_gen, btn_solve],
           layout=w.Layout(align_items="center",gap="8px",margin="0 0 8px 0",flex_wrap="wrap")),
    w.HBox([btn_bench, btn_levels],
           layout=w.Layout(align_items="center",gap="8px",margin="0 0 10px 0")),
    tab
], layout=w.Layout(max_width="880px")))

on_generate(None)
